<a href="https://colab.research.google.com/github/machine-perception-robotics-group/MPRGDeepLearningLectureNotebook/blob/develop-v2/99_others/Introduction_to_Optuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Optuna入門

---
## 目的
ハイパーパラメータ自動最適化フレームワーク Optuna の使い方を学ぶ．具体的には，(1) ランダムフォレストのハイパーパラメータ探索と，(2) PyTorchで実装した多層パーセプトロン（MLP）のハイパーパラメータ探索という2つの実践的な題材を通して，Optunaの基本的なAPIと，枝刈り（Pruning）や可視化などの実用的な機能を習得する．

## ハイパーパラメータ探索とは
機械学習・深層学習のモデルには，学習によって自動的に決まる「パラメータ」（重みなど）とは別に，学習を始める前に人間が値を決めておく必要がある「ハイパーパラメータ」が存在する．例えば，ニューラルネットワークの層数やユニット数，学習率，バッチサイズ，ランダムフォレストの木の数や深さなどがハイパーパラメータにあたる．

ハイパーパラメータの良し悪しはモデルの性能に大きく影響するが，理論的に最適な値を一発で求めることは難しく，多くの場合は値を変えながら何度も学習・評価を繰り返して良い組み合わせを探す必要がある．この作業を**ハイパーパラメータ探索（ハイパーパラメータチューニング）**と呼ぶ．

手作業で1つずつ値を変えて試す方法（マニュアルサーチ）や，あらかじめ決めた候補を総当たりで試す方法（グリッドサーチ）は，組み合わせ数が増えると膨大な時間がかかってしまい非効率である．

## Optunaとは
Optunaは，株式会社Preferred Networksが開発したオープンソースのハイパーパラメータ自動最適化フレームワークである．PyTorchやTensorFlow，scikit-learnなど，様々な機械学習・深層学習フレームワークと組み合わせて使用できる．主な特徴は以下の通りである．

- **Define-by-Runインターフェース**: 探索するハイパーパラメータの範囲や種類を，通常のPythonコードを書く感覚で動的に定義できる．
- **効率的な探索アルゴリズム**: ベイズ最適化の一種であるTPE（Tree-structured Parzen Estimator）などのアルゴリズムにより，過去の試行結果を活用して次に試すべき値を賢く選択する．ランダムサーチやグリッドサーチよりも少ない試行回数で良い結果に近づきやすい．
- **枝刈り（Pruning）**: 学習の途中経過を見て，明らかに良い結果が期待できない試行を早期に打ち切ることで，計算時間を節約できる．
- **可視化機能**: 探索の過程やハイパーパラメータの重要度をグラフで確認できる．

## このノートブックの流れ
1. Optunaのインストール
2. Optunaの基本的な使い方（Study・Trial・Objective関数）
3. 実践1：ランダムフォレストのハイパーパラメータ探索（scikit-learn）
4. 実践2：ニューラルネットワーク（MLP）のハイパーパラメータ探索（PyTorch）と枝刈り
5. 探索結果の可視化

## 準備：Optunaのインストール
Google Colaboratoryには標準ではOptunaがインストールされていないため，`pip`コマンドでインストールする．本ノートブックでは，執筆時点（2026年7月）で最新のバージョンである Optuna 4.9.0 を使用する．

In [ ]:
!pip install -q optuna==4.9.0

## Optunaの基本的な使い方
Optunaでは，主に次の3つの要素を組み合わせてハイパーパラメータ探索を記述する．

- **Study**: 最適化タスク全体を管理するオブジェクト．`optuna.create_study()`で作成し，`study.optimize()`で探索を実行する．
- **Trial**: 1回分のハイパーパラメータの試行を表すオブジェクト．`objective`関数の引数として渡される．
- **Objective関数**: 最小化（または最大化）したい評価値を返す関数．関数の中で`trial.suggest_*`メソッドを使ってハイパーパラメータの値を提案（サンプリング）し，その値でモデルの学習・評価を行い，評価値（誤り率や精度など）を返す．

代表的な`suggest_*`メソッドは以下の通りである．

| メソッド | 用途 | 使用例 |
| --- | --- | --- |
| `trial.suggest_float(name, low, high)` | 連続値（浮動小数点数）を提案する | 学習率，ドロップアウト率など |
| `trial.suggest_int(name, low, high)` | 整数値を提案する | 層数，ユニット数，木の数など |
| `trial.suggest_categorical(name, choices)` | 離散的な選択肢の中から1つを提案する | オプティマイザの種類，活性化関数の種類など |

これらのメソッドをobjective関数の中で自由に組み合わせることで，様々な種類・範囲のハイパーパラメータを柔軟に定義できる．これがOptunaの特徴であるDefine-by-Runインターフェースである．

## 実践1：ランダムフォレストのハイパーパラメータ探索
最初の例として，scikit-learnの`RandomForestClassifier`のハイパーパラメータをOptunaで探索する．題材には，8×8ピクセルの手書き数字画像データセット`digits`（10クラス，1,797サンプル）を使用する．学習にかかる時間が短いため，Optunaの基本的な使い方を確認するのに適した題材である．

ランダムフォレストは，複数の決定木を組み合わせて識別を行うアンサンブル学習器であり，主に以下のハイパーパラメータを持つ．

- `n_estimators`: 決定木の数
- `max_depth`: 決定木の最大の深さ
- `min_samples_split`: ノードを分割するために必要な最小サンプル数
- `criterion`: 分割の良さを評価する指標（`gini`または`entropy`）

これらを手動で1つずつ調整するのは手間がかかるため，Optunaを用いて自動的に良い組み合わせを探索する．

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import optuna

# データセットの読み込み（8x8の手書き数字画像，10クラス）
digits = load_digits()
X, y = digits.data, digits.target
print('入力データの形状:', X.shape)
print('クラス数:', len(np.unique(y)))

def objective_rf(trial):
    # 探索するハイパーパラメータの範囲を定義する
    n_estimators = trial.suggest_int('n_estimators', 10, 200)
    max_depth = trial.suggest_int('max_depth', 2, 32, log=True)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 16)
    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy'])

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        criterion=criterion,
        random_state=0,
        n_jobs=-1,
    )

    # 5分割交差検証で精度を評価し，その平均値をOptunaに返す
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    return scores.mean()

study_rf = optuna.create_study(direction='maximize', study_name='random_forest_digits')
study_rf.optimize(objective_rf, n_trials=50)

print('試行回数:', len(study_rf.trials))
print('最良の精度:', study_rf.best_value)
print('最良のハイパーパラメータ:', study_rf.best_params)

## 実践2：ニューラルネットワーク（MLP）のハイパーパラメータ探索
次に，より実践に近い題材として，PyTorchで実装した多層パーセプトロン（MLP）によるMNIST手書き数字認識のハイパーパラメータをOptunaで探索する．

探索するハイパーパラメータは以下の通りである．

- 中間層の数（`n_layers`）
- 各層のユニット数（`n_units_l{i}`）
- ドロップアウト率（`dropout_rate`）
- オプティマイザの種類（`optimizer`）
- 学習率（`learning_rate`）

ニューラルネットワークの学習には時間がかかるため，学習データの一部のみを使用し，エポック数も少なく設定することで，1回の試行（Trial）にかかる時間を短縮する．

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('使用デバイス:', device)

transform = transforms.Compose([transforms.ToTensor()])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# 1回の試行にかかる時間を短縮するため，学習・評価データの一部のみを使用する
rng = np.random.RandomState(0)
train_indices = rng.choice(len(train_dataset), 5000, replace=False)
test_indices = rng.choice(len(test_dataset), 1000, replace=False)

train_loader = DataLoader(Subset(train_dataset, train_indices), batch_size=128, shuffle=True)
test_loader = DataLoader(Subset(test_dataset, test_indices), batch_size=256, shuffle=False)

print('学習に使用するサンプル数:', len(train_indices))
print('評価に使用するサンプル数:', len(test_indices))

In [ ]:
def create_model(trial):
    # 中間層の数を探索する
    n_layers = trial.suggest_int('n_layers', 1, 3)
    dropout_rate = trial.suggest_float('dropout_rate', 0.0, 0.5)

    layers = [nn.Flatten()]
    in_features = 28 * 28
    for i in range(n_layers):
        # 各層のユニット数を探索する
        n_units = trial.suggest_int(f'n_units_l{i}', 16, 128, log=True)
        layers.append(nn.Linear(in_features, n_units))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout_rate))
        in_features = n_units
    layers.append(nn.Linear(in_features, 10))

    return nn.Sequential(*layers)

## 枝刈り（Pruning）による探索の効率化
ニューラルネットワークの学習には時間がかかるため，明らかに精度が伸びそうにない試行はできるだけ早く打ち切りたい．Optunaにはこのための**枝刈り（Pruning）**という機能がある．

枝刈りを利用するには，各エポックの終了時に`trial.report(評価値, step)`で学習の途中経過をOptunaに報告し，続けて`trial.should_prune()`を呼び出す．これが`True`を返した場合は`optuna.TrialPruned()`例外を送出して学習を打ち切ることで，見込みのない試行に時間を使わないようにできる．

ここでは，それまでの試行の中央値と比較して成績が悪い試行を打ち切る`MedianPruner`を使用する．

In [ ]:
def objective_mlp(trial):
    model = create_model(trial).to(device)

    # オプティマイザの種類と学習率を探索する
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'SGD', 'RMSprop'])
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True)
    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=learning_rate)

    criterion = nn.CrossEntropyLoss()
    n_epochs = 5

    for epoch in range(n_epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        # 検証用データで精度を計算する
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                predicted = outputs.argmax(dim=1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        accuracy = correct / total

        # 途中経過をOptunaに報告し，見込みのない試行は打ち切る
        trial.report(accuracy, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return accuracy

study_mlp = optuna.create_study(
    direction='maximize',
    study_name='mlp_mnist',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)
study_mlp.optimize(objective_mlp, n_trials=20)

pruned_trials = [t for t in study_mlp.trials if t.state == optuna.trial.TrialState.PRUNED]
complete_trials = [t for t in study_mlp.trials if t.state == optuna.trial.TrialState.COMPLETE]

print('試行回数:', len(study_mlp.trials))
print('枝刈りされた試行数:', len(pruned_trials))
print('完了した試行数:', len(complete_trials))
print('最良の精度:', study_mlp.best_value)
print('最良のハイパーパラメータ:', study_mlp.best_params)

## 探索結果の可視化
`optuna.visualization`モジュールを使うと，探索結果をインタラクティブなグラフとして確認できる．

- `plot_optimization_history`: 試行ごとの評価値の推移を確認できる．
- `plot_param_importances`: 各ハイパーパラメータが評価値にどの程度影響しているかを確認できる．
- `plot_slice`: 各ハイパーパラメータの値と評価値の関係を確認できる．

In [ ]:
optuna.visualization.plot_optimization_history(study_mlp)

In [ ]:
optuna.visualization.plot_param_importances(study_mlp)

In [ ]:
optuna.visualization.plot_slice(study_mlp)

## まとめ
本ノートブックでは，Optunaを用いたハイパーパラメータ自動探索の基本的な流れを，(1) ランダムフォレストと(2) PyTorchによるMLPという2つの実践的な題材を通して学んだ．手動での試行錯誤やグリッドサーチに比べて，Optunaを用いることで少ない試行回数で効率的に良いハイパーパラメータの組み合わせを見つけることができる．また，枝刈り（Pruning）を活用することで，見込みのない試行にかかる計算時間を節約できることも確認した．

## 課題
1. ランダムフォレストの探索範囲（`n_estimators`や`max_depth`など）を変更すると，最良の精度はどのように変化するか確認せよ．
2. MLPの`n_trials`や`n_epochs`を増やすと，探索結果や実行時間はどのように変化するか確認せよ．
3. サンプラーをデフォルトの`TPESampler`から`optuna.samplers.RandomSampler`に変更し，`n_trials`が同じ場合の探索結果を比較せよ．

## ヒント
1. `max_depth`や`n_estimators`を大きくしすぎると，学習・評価に時間がかかるようになる．
2. `n_trials`を増やすほど良いハイパーパラメータが見つかりやすくなるが，その分実行時間も増加する．枝刈りを併用すると増加を緩和できる．
3. サンプラーは`optuna.create_study(sampler=optuna.samplers.RandomSampler())`のように指定する．